In [ ]:
import sys
import os
import torch
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from PIL import Image
from huggingface_hub import login

In [ ]:
# sanity check: print path to the Python binary, verify Python version
print(sys.executable, sys.version_info)
assert sys.version_info.major == 3 and sys.version_info.minor == 12, "Python version does not match the expected version."

In [ ]:
# huggingface login
login()

In [ ]:
import sam3
from sam3 import build_sam3_image_model
from sam3.model.box_ops import box_xywh_to_cxcywh
from sam3.model.sam3_image_processor import Sam3Processor
from sam3.visualization_utils import draw_box_on_image, normalize_bbox, plot_results

sam3_root = os.path.join(os.path.dirname(sam3.__file__), "..")

In [ ]:
sys.path.append(os.path.abspath(".."))

from Util.env_utils import load_segmentation_env

In [ ]:
load_segmentation_env()

RAW_DATA_DIR = os.getenv("RAW_DATA_DIR")
HIGHRES_IMG_DIR_NAME = os.getenv("HIGHRES_IMG_DIR_NAME", "images_4096")
SAM3_OUTPUT_DIR_NAME = os.getenv("SAM3_OUTPUT_DIR_NAME", "sam3_outputs")

EXPORT_MASKS = False

In [ ]:
# turn on tfloat32 for Ampere GPUs
# https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# use bfloat16 for the entire notebook
torch.autocast("cuda", dtype=torch.bfloat16).__enter__()

In [ ]:
bpe_path = f"{sam3_root}/assets/bpe_simple_vocab_16e6.txt.gz"
model = build_sam3_image_model(bpe_path=bpe_path)

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"SAM3 total parameters: {total_params}")
print(f"SAM3 trainable parameters: {trainable_params} \n")

In [ ]:
device = torch.device("cuda:0")
total_memory = torch.cuda.get_device_properties(device).total_memory
print(f"GPU total memory: {total_memory / (1024**3):.2f} GB")

In [ ]:
resize_dims = (1024, 1024)


def zeroshot_segmentation_textprompt(image_path, prompt="lines"):
    image = Image.open(image_path).resize(resize_dims)

    processor = Sam3Processor(model, confidence_threshold=0.5)
    inference_state = processor.set_image(image)

    processor.reset_all_prompts(inference_state)
    output = processor.set_text_prompt(state=inference_state, prompt=prompt)
    masks = output["masks"]

    # convert image (for plotting) to 512x512 RGB (RGB (3 channels) conversion needed because otherwise will look green in plot)
    img0 = Image.open(image_path).resize(resize_dims).convert("RGB")
    plot_results(img0, inference_state)
    return masks.cpu()

In [ ]:
images_dir = Path(RAW_DATA_DIR) / HIGHRES_IMG_DIR_NAME

masks = {}

for img_path in images_dir.glob("*.png"):
    img_name = img_path.name
    print(img_name)
    
    output_masks = zeroshot_segmentation_textprompt(str(img_path), prompt="lines")
    masks[img_name] = output_masks.any(dim=0).numpy().astype(np.uint8)

In [ ]:
sam_output_dir = Path(RAW_DATA_DIR) / SAM3_OUTPUT_DIR_NAME


def export_np_mask(mask: np.ndarray, filename: str):
    mask_2d = mask.squeeze().astype(np.uint8) * 255
    img_mask = Image.fromarray(mask_2d, mode="L")
    img_mask = img_mask.resize((4096, 4096), resample=Image.NEAREST)
    img_mask.save(sam_output_dir / filename, format="PNG")


if EXPORT_MASKS:
    for filename, mask in masks.items():
        export_np_mask(mask, filename)

In [ ]:
zeroshot_segmentation_textprompt('/data/jhehli/additional_data/20240425_andrea_lila_soi_2_bottom.png')